# RouteAlpha 分阶段 Walkthrough

| 阶段 | 内容 |
|------|------|
| **数据导入** | 加载 RouterBench peek 子集 |
| **阶段一** | XGB 预测 + OOF 回测 + 概率校准 |
| **阶段二** | MILP 预算约束路由 + Pareto 迭代 |
| **失败归因** | 五类 badcase 分析 |
| **LLM-judge** | 位置偏置 + swap-and-aggregate |


In [ ]:
# 数据导入: RouterBench peek 子集 (见 data/peek.csv)
import pandas as pd
from model import ml_seperate as ml

cfg = ml.load_config("config/config.yaml")
peek = pd.read_csv(cfg["data"]["csv_path"], encoding="gb18030", nrows=5)
print(f"数据路径: {cfg['data']['csv_path']} | max_samples={cfg['data'].get('max_samples')}")
peek.head()


## 阶段一 · 预测器 + 概率校准

每 model 独立 XGB：`bge-small` embedding + 结构特征 → P(success)。OOF 回测无穿越；Isotonic 校准降低 ECE。


In [ ]:
import importlib
from model import ml_seperate as ml
importlib.reload(ml)

cfg = ml.load_config("config/config.yaml")
# 测试时可按需调小提速；正式跑直接用 config 里的值
# cfg["data"]["max_samples"] = 800

data = ml.load_data(cfg)
feat = ml.Featurizer(cfg, prompts=data.prompts, eval_names=data.eval_names)
if feat.use_embedding:
    feat._ensure_backend(data.prompts)
    if feat.is_fittable:
        print("特征后端 (TF-IDF, 回测时按折 fit):", feat.used_backend_)
    else:
        X = feat.encode_frozen(data.prompts)
        print("语义特征矩阵:", X.shape, "| backend =", feat.used_backend_)
print("选定模型:", data.models)
fc = cfg.get("features", {})
print(
    "特征开关:",
    f"TE={fc.get('use_target_encoding', False)}",
    f"cross={fc.get('use_cross_difficulty', False)}",
    f"alpha={fc.get('target_encoding_alpha', 10)}",
    f"| max_samples={cfg['data'].get('max_samples')}",
)

## 阶段一 · 1.2 滚动回测 & 考核指标

产出长表 `pred_df`：每行 = 一条 query × 一个 model，含 `y_true` 与 `p_success`。

| 指标 | 含义 |
|------|------|
| accuracy | @0.5 分类准确率（主考核） |
| auc | 排序能力：成功题 p 是否高于失败题 |
| brier | 概率 MSE |
| ece | 校准误差：预测概率 vs 实际成功率 |

In [ ]:
# 阶段一 · 1.2  滚动回测 + 考核指标
# 注意: 第二个参数是 feat (Featurizer)，不是特征矩阵 X
pred_df = ml.rolling_backtest(cfg, feat, data)
metrics_df = ml.compute_metrics(pred_df)  # accuracy / auc / brier / ece

ml.save_predictions(cfg, pred_df, metrics_df)
print(f"预测行数: {len(pred_df)} | query 数: {pred_df['sample_id'].nunique()}")
metrics_df.round(4)



## 阶段一 · 结果：可靠性图 + 模型质量


In [ ]:
# 阶段一 · 1.3  可靠性图 + 指标对比
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.calibration import calibration_curve


def setup_chinese_font() -> str | None:
    """macOS/Windows 常见中文字体; 找不到则回退英文标签。"""
    candidates = [
        "PingFang SC",
        "Heiti SC",
        "STHeiti",
        "Arial Unicode MS",
        "SimHei",
        "Microsoft YaHei",
    ]
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in candidates:
        if name in available:
            plt.rcParams["font.sans-serif"] = [name, "DejaVu Sans"]
            plt.rcParams["axes.unicode_minus"] = False
            return name
    plt.rcParams["axes.unicode_minus"] = False
    return None


_font = setup_chinese_font()
if _font:
    print(f"matplotlib 中文字体: {_font}")
else:
    print("未找到中文字体, 图表标签可能显示为方框")

def _overall_ece(y_true, p, n_bins: int = 10) -> float:
    """按分位数分箱的整体 ECE，和可靠性曲线口径一致。"""
    p = np.asarray(p, dtype=float)
    y_true = np.asarray(y_true, dtype=float)
    edges = np.unique(np.quantile(p, np.linspace(0, 1, n_bins + 1)))
    ece, n = 0.0, len(p)
    for i in range(len(edges) - 1):
        lo, hi = edges[i], edges[i + 1]
        mask = (p >= lo) & (p <= hi) if i == len(edges) - 2 else (p >= lo) & (p < hi)
        if mask.sum():
            ece += abs(y_true[mask].mean() - p[mask].mean()) * mask.sum() / n
    return ece


y_true = pred_df["y_true"].to_numpy(dtype=float)
p_cal = pred_df["p_success"].to_numpy(dtype=float)
has_raw = "p_success_raw" in pred_df.columns
p_raw = pred_df["p_success_raw"].to_numpy(dtype=float) if has_raw else None
ece_cal = _overall_ece(y_true, p_cal)
ece_raw = _overall_ece(y_true, p_raw) if has_raw else None

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# ── 左：可靠性图（校准前 → 校准后）────────────────────────────
ax0 = axes[0]
# 背景：预测概率分布（说明各分箱的样本密度，避免"点很少"的误解）
axb = ax0.twinx()
axb.hist(p_cal, bins=20, range=(0, 1), color="#cfcfcf", alpha=0.35, zorder=0)
axb.set_ylabel("样本数（校准后分布）", color="#9e9e9e", fontsize=9)
axb.tick_params(axis="y", labelcolor="#9e9e9e", labelsize=8)
ax0.set_zorder(axb.get_zorder() + 1)
ax0.patch.set_visible(False)

ax0.plot([0, 1], [0, 1], "k--", linewidth=1.0, label="完美校准", zorder=2)
if has_raw:
    fr_raw, mp_raw = calibration_curve(y_true, p_raw, n_bins=10, strategy="quantile")
    ax0.plot(mp_raw, fr_raw, "o-", color="#f16913", linewidth=1.6, markersize=5,
             label=f"校准前 raw (ECE={ece_raw:.3f})", zorder=3)
fr_cal, mp_cal = calibration_curve(y_true, p_cal, n_bins=10, strategy="quantile")
ax0.plot(mp_cal, fr_cal, "o-", color="#2171b5", linewidth=1.8, markersize=5,
         label=f"校准后 cal (ECE={ece_cal:.3f})", zorder=4)
ax0.set_xlim(0, 1)
ax0.set_ylim(0, 1)
ax0.set_xlabel("预测成功率")
ax0.set_ylabel("实际成功率")
title0 = "可靠性图 · 校准前 → 校准后" if has_raw else "可靠性图（校准后）"
ax0.set_title(title0)
ax0.legend(loc="upper left", fontsize=9, framealpha=0.9)
ax0.grid(alpha=0.3)

# ── 右：各候选模型预测质量 ────────────────────────────────
ax1 = axes[1]
m = metrics_df[metrics_df["model"] != "__overall__"].copy()
name_map = {
    "gpt-4-1106-preview": "GPT-4",
    "gpt-3.5-turbo-1106": "GPT-3.5",
    "claude-v2": "Claude-v2",
    "claude-instant-v1": "Claude-Instant",
}
labels = [name_map.get(s, s.split("/")[-1][:12]) for s in m["model"]]
x = np.arange(len(m))
w = 0.26
bars = [
    ax1.bar(x - w, m["accuracy"], w, label="accuracy", color="#4c78a8"),
    ax1.bar(x, m["auc"], w, label="AUC", color="#f58518"),
    ax1.bar(x + w, 1 - m["ece"], w, label="1−ECE", color="#54a24b"),
]
ax1.axhline(0.5, color="#999999", linestyle="--", linewidth=1.0, alpha=0.7,
            label="AUC 随机基线 0.5")
for group in bars:
    for b in group:
        ax1.annotate(f"{b.get_height():.2f}", (b.get_x() + b.get_width() / 2, b.get_height()),
                     textcoords="offset points", xytext=(0, 2), ha="center", fontsize=7)
ax1.set_xticks(x)
ax1.set_xticklabels(labels, fontsize=9)
ax1.set_ylim(0, 1.08)
ax1.set_ylabel("指标值（越高越好）")
ax1.set_title("各候选模型预测质量（阶段一）")
ax1.legend(fontsize=7.5, ncol=4, loc="upper center")
ax1.grid(axis="y", alpha=0.3)

plt.tight_layout()
Path("images").mkdir(exist_ok=True)
fig.savefig("images/calibration.png", dpi=150, bbox_inches="tight")
plt.show()
print("已保存 images/calibration.png")



## 阶段二 · MILP 预算约束路由


In [ ]:
# MILP 预算约束路由 vs 四条 baseline
from model import milp as mp
importlib.reload(mp)

budget_per_query = cfg["milp"]["budget_per_query"]
table = mp.compare(pred_df, budget_per_query=budget_per_query)
table

## 阶段二 · 迭代记录 & Pareto 进步图


In [ ]:
# 阶段二 · 2.3  迭代记录：把当前 pred_df 的成绩记一行（不是特征开关！）
# 特征开关在 config.yaml → features.*；改 config 后须重跑 rolling_backtest 再调本 cell
from pathlib import Path

import pandas as pd
from model import milp as mp

ITER_LOG = Path("data/iteration_log.csv")


def feature_note(cfg: dict) -> str:
    """从 config 自动生成备注，避免 version/note 与真实开关不一致。"""
    fc = cfg.get("features", {})
    parts = ["bge384"]
    if fc.get("use_structural", True):
        parts.append("24维结构")
    if fc.get("use_target_encoding", False):
        parts.append(f"TE(alpha={fc.get('target_encoding_alpha', 10)})")
    if fc.get("use_cross_difficulty", False):
        parts.append("cross+mglobal")
    return " + ".join(parts)


def record_iteration(version: str, pred_df, cfg, note: str = "", prob_col: str = "p_success") -> pd.DataFrame:
    """记录当前版本在固定预算下的成绩，持久化到 csv（同名版本覆盖）。"""
    budget = cfg["milp"]["budget_per_query"]
    Q = pred_df["sample_id"].nunique()
    metrics = ml.compute_metrics(pred_df, prob_col=prob_col)
    ov = metrics[metrics["model"] == "__overall__"].iloc[0]
    oracle_s = mp.baselines(pred_df, prob_col=prob_col)["oracle"].realized_success_rate
    milp = mp.solve_routing(pred_df, budget_per_query=budget, prob_col=prob_col)
    row = {
        "版本": version,
        "AUC": round(float(ov["auc"]), 4),
        "ECE": round(float(ov["ece"]), 4),
        "MILP成功率": round(milp.realized_success_rate, 4),
        "成本/q": round(milp.total_cost / Q, 5),
        "oracle天花板": round(oracle_s, 4),
        "gap": round(oracle_s - milp.realized_success_rate, 4),
        "预算/q": budget,
        "备注": note or feature_note(cfg),
    }
    if ITER_LOG.exists():
        log = pd.read_csv(ITER_LOG, encoding="utf-8")
        log = log[log["版本"] != version]
        log = pd.concat([log, pd.DataFrame([row])], ignore_index=True)
    else:
        ITER_LOG.parent.mkdir(parents=True, exist_ok=True)
        log = pd.DataFrame([row])
    log.to_csv(ITER_LOG, index=False, encoding="utf-8")
    return log


# 换实验版本：改 config → 重跑 rolling_backtest → 改 version 字符串 → 跑本 cell
# 同名 version 会覆盖旧行；不同 version 会累积多行供 2.3b 画图
iter_log = record_iteration(
    "v3_TE+cross+mglobal",
    pred_df,
    cfg,
    # note 留空则自动从 config 生成
)
iter_log

In [ ]:
# 阶段二 · 2.3b  迭代进步图（极简 · v1→v2→v3 @ 0.001/query）
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from model import milp as mp

if "setup_chinese_font" in globals():
    setup_chinese_font()

DEMO_BUDGET = 0.0010
VERSIONS = [
    ("v1", Path("data/predictions_v0.parquet"), "#bdbdbd", "v1 · v0_baseline"),
    ("v2", Path("data/predictions_v2.parquet"), "#6baed6", "v2 · v2_TE_only"),
    ("v3", Path("data/predictions.parquet"), "#08519c", "v3 · v3_TE+cross+mglobal"),
]

ref = pd.read_parquet(VERSIONS[-1][1])
Q = ref["sample_id"].nunique()
bs = mp.baselines(ref)

def cpq(r):
    return r.total_cost / Q, r.realized_success_rate

fig, ax = plt.subplots(figsize=(6.5, 5.5))

# baseline：oracle / always-cheap / always-expensive / random
for key, c, m, s, lab in [
    ("oracle", "#ffbf00", "*", 120, "oracle"),
    ("always_cheap", "#2ca02c", "s", 40, "always-cheap"),
    ("always_expensive", "#d62728", "^", 40, "always-expensive"),
    ("random", "#9467bd", "D", 40, "random"),
]:
    cx, sy = cpq(bs[key])
    ax.scatter(cx, sy, c=c, marker=m, s=s, label=lab, edgecolors="k", linewidths=0.4, zorder=5)

# v1 / v2 / v3：同预算、同成本竖排；SR 值直接写在点旁（错开避免叠字）
traj_x, traj_y = [], []
MARKER_S = {"v1": 15, "v2": 15, "v3": 15}
LABEL_OFFSET = {
    "v1": (-12, -13),   # 左下
    "v2": (12, -7),     # 右略下
    "v3": (12, 13),     # 右上
}
for short, path, color, legend_lab in VERSIONS:
    pred = pd.read_parquet(path)
    milp = mp.solve_routing(pred, budget_per_query=DEMO_BUDGET)
    cx, sy = cpq(milp)
    traj_x.append(cx)
    traj_y.append(sy)
    ax.scatter(
        cx, sy,
        s=MARKER_S[short],
        c=color,
        label=legend_lab,
        edgecolors="k",
        linewidths=0.6,
        zorder=8,
    )
    ox, oy = LABEL_OFFSET[short]
    ax.annotate(
        f"{short}  {sy:.3f}",
        (cx, sy),
        xytext=(ox, oy),
        textcoords="offset points",
        fontsize=8.5,
        color=color if short == "v3" else "black",
        fontweight="bold" if short == "v3" else "normal",
        ha="right" if short == "v1" else "left",
        va="center",
    )

x_hi = cpq(bs["always_expensive"])[0] + 0.00012
ax.set_xlim(0, x_hi)
ax.set_ylim(0.4, 0.95)

# v1–v3 固定预算：竖虚线 + 成本数字
ax.axvline(DEMO_BUDGET, color="#2171b5", linestyle=":", linewidth=1.1, alpha=0.7, zorder=1)
y_lo = ax.get_ylim()[0]
ax.text(
    DEMO_BUDGET,
    y_lo + 0.006,
    f"{DEMO_BUDGET:.3f}",
    fontsize=8,
    ha="center",
    va="bottom",
    color="#2171b5",
)

# 迭代轨迹：同成本下竖直虚线 v1→v2→v3
ax.plot(traj_x, traj_y, "--", color="#2171b5", linewidth=1.4, alpha=0.75, zorder=6)

# 参考斜虚线：左下→右上（背景示意）；箭头单独指左上 = 低成本+高成功率
x_lo, x_hi_plot = ax.get_xlim()
y_lo, y_hi_plot = ax.get_ylim()
ax.plot(
    [x_lo + 0.00003, x_hi_plot * 0.88],
    [y_lo + 0.008, y_hi_plot - 0.008],
    linestyle="--",
    color="#888888",
    linewidth=1.0,
    alpha=0.45,
    zorder=0,
)
ax.text(
    x_lo + (x_hi_plot - x_lo) * 0.38,
    y_lo + (y_hi_plot - y_lo) * 0.52,
    "更优方向 ↖",
    fontsize=8,
    color="#999999",
    rotation=-27,
    rotation_mode="anchor",
    ha="center",
    va="bottom",
)

ax.set_xlabel("平均成本 / query")
ax.set_ylabel("真实成功率")
ax.set_title(f"迭代进步（Pareto）· {DEMO_BUDGET}/query · v1→v2→v3")
ax.legend(
    loc="lower right",
    fontsize=8.5,
    markerscale=1.1,
    labelspacing=0.45,
    handletextpad=0.6,
    borderpad=0.5,
    framealpha=0.9,
)
plt.tight_layout()
Path("images").mkdir(exist_ok=True)
fig.savefig("images/pareto_iteration.png", dpi=150, bbox_inches="tight")
plt.show()
print("已保存 images/pareto_iteration.png")

## 失败样本归因 (badcase)


In [ ]:
# 阶段二 · 2.4  失败样本归因
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))
from eval.failure_analysis import analyze_routing_failures, plot_failure_summary, write_report

if "setup_chinese_font" in globals():
    setup_chinese_font()

budget_attr = cfg["milp"]["budget_per_query"]
badcase_df, summary_df, attr_metrics = analyze_routing_failures(
    pred_df,
    budget_per_query=budget_attr,
    max_rows=15,
)
attr_metrics["budget_per_query"] = budget_attr

display(summary_df)
display(badcase_df)

fig, ax = plt.subplots(figsize=(6.5, 3.8))
plot_failure_summary(summary_df, ax=ax)
plt.tight_layout()

Path("images").mkdir(exist_ok=True)
fig.savefig("images/failure_attribution.png", dpi=150, bbox_inches="tight")
print("已保存 images/failure_attribution.png")

write_report(badcase_df, summary_df, attr_metrics, Path("eval/failure_report.md"))
print("指标:", attr_metrics)
print("报告已写入 eval/failure_report.md")

## LLM-as-Judge · 位置偏置诊断

swap-and-aggregate：正反各评一次，仅一致才采纳。下方展示 mock 对比 + 已提交的 Kimi 真实结果。


In [ ]:
from pathlib import Path
# 评测 · LLM-as-judge 位置偏置诊断 (离线 mock + Kimi 真实结果)
import json
import pandas as pd
from eval.judge import DEMO_PAIRS, make_mock_judge, run_judge_eval

fair = run_judge_eval(make_mock_judge(position_bias=0.0, seed=42), DEMO_PAIRS)
biased = run_judge_eval(make_mock_judge(position_bias=5.0, seed=42), DEMO_PAIRS)

print("===== mock: 公平 judge =====")
print(fair["metrics"])
print("===== mock: 有位置偏置 judge =====")
print(biased["metrics"])

kimi_path = Path("eval/kimi_judge_results.json")
if kimi_path.exists():
    kimi = json.loads(kimi_path.read_text(encoding="utf-8"))
    print(f"\n===== Kimi 真实 judge ({kimi['meta']['model']}) =====")
    print(kimi["metrics"])
    print("分难度:", kimi["by_category"])
    display(pd.DataFrame(kimi["rows"]))
else:
    print("\n未找到 eval/kimi_judge_results.json; 运行 python eval/judge.py --real 可生成")
